# Modelos finales de las pistas M y S

Reentrena el mejor modelo de cada pista según el test gold (`todo_cob`, F1-macro) de los notebooks 08 y 09, guarda sus pesos
y los vuelve a evaluar sobre el gold. Los datos deben ser exactamente los de 08/09 (se comprueba con el sha de los ids).

| Pista | Corrida | Estrategia | lr | PH | Entrada | Semilla |
|---|---|---|---|---|---|---|
| M | `M-AB-E0-lr3e-05-s2026` | E0 | 3e-05 | sí | AB | 2026 |
| S | `S-AB-E3-lr2e-05-s42` | E3 | 2e-05 | no | AB | 42 |

El Top-3 no se recalcula: se lee de `experimentos_08_09/<pista>/top3_v2.json` y se verifica con la huella de los chunks.

In [1]:
import gc
import hashlib
import json
import math
import os
import platform
import random
import re
import time
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
import torch
import transformers
from safetensors.torch import load_file, save_file
from torch import nn
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

/home/carlos/microproyecto-local-citation/local_workflows/.venv-qlora/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 0. Configuración

In [2]:
# ---- modelos a reentrenar (mejor corrida de 08 / 09 en el gold, todo_cob)
MODELOS = {
    "M": {"estrategia": "E0", "lr": 3e-5, "ph": True, "semilla": 2026, "entrada": "AB"},
    "S": {"estrategia": "E3", "lr": 2e-5, "ph": False, "semilla": 42, "entrada": "AB"},
}
PISTAS = {
    "M": {"train": "natural.csv", "v": "validacion.csv", "kd": True},
    "S": {"train": "sonnet.csv", "v": "validacion_sonnet.csv", "kd": False},
}

# ---- rutas
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATOS = REPO / "proyecto_de_grado/artifacts/experimentos_104/datos"
RAW = REPO / "data/raw"
CACHE_OCL = REPO / "proyecto_de_grado/scripts/spike_enlace_datos/cache_ocl"
PREDS_QLORA = [REPO / "local_workflows/qlora_18000_preds_v2.jsonl",
               *[REPO / f"local_workflows/subconjuntos_qlora/CITAS_NO_USADAS_0{i}_qlora_v2_preds.jsonl" for i in range(1, 6)]]
GOLD_CSV = REPO / "proyecto_de_grado/anotacion/test_gold_final/test_gold_final.csv"
GOLD_SHA_PREFIJO = "80e4d9a6"
HF_HOME = Path(os.environ.get("HF_HOME", Path.home() / ".cache/huggingface"))
SCIBERT = HF_HOME / "hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1"

SALIDA = REPO / "proyecto_de_grado/artifacts/experimentos_10"
EVIDENCIAS = REPO / "proyecto_de_grado/docs/evidencias_experimentos_10"
ORIGEN_08_09 = REPO / "proyecto_de_grado/artifacts/experimentos_08_09"       # json, npz y top3_v2.json de referencia
EVIDENCIAS_08_09 = REPO / "proyecto_de_grado/docs/evidencias_experimentos_08_09"
for carpeta in (SALIDA, EVIDENCIAS):
    carpeta.mkdir(parents=True, exist_ok=True)

# ---- taxonomia
CLASES = ("Background", "Gap", "Basis", "Comparison", "Application", "Improvement / Modification", "Evidence",
          "Identification of the Originator", "Further Reading")
N_CLASES = len(CLASES)
INDICE = {c: i for i, c in enumerate(CLASES)}

# ---- datos y controles
FRACCION_DEV = 0.15
SAL_DEV = "dev-08-09"
K_SHINGLE = 8
UMBRAL_SOLAPE = 0.5

# ---- fragmentacion y entrada
MAX_PALABRAS = 300
MAX_PARRAFOS_CHUNK = 2
MAX_CARACTERES = 5000
TOP_K = 3
MAX_LEN = 512
TOPE_CONTEXTO = 192
TOPE_TITULO = 48

# ---- entrenamiento
EPOCAS = 4
PACIENCIA = 2
BATCH = 8
ACUMULACION = 2
WARMUP = 0.1
WEIGHT_DECAY = 0.01
CLIP = 1.0
DROPOUT = 0.1
BATCH_INFERENCIA = 32
GAMMA_FOCAL = 2.0
BETA_CB = 0.999
GRILLA_SESGOS = np.round(np.arange(-2.0, 2.0001, 0.1), 2)
RONDAS_SESGOS = 3

# ---- determinismo (antes de la primera operacion CUDA)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ENTORNO = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
           "transformers": transformers.__version__, "sklearn": sklearn.__version__, "numpy": np.__version__,
           "pandas": pd.__version__, "scipy": scipy.__version__,
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "scibert_snapshot": SCIBERT.name}
assert SCIBERT.is_dir(), f"Falta el snapshot local de SciBERT: {SCIBERT}"
print(json.dumps(ENTORNO, indent=2))

{
  "python": "3.12.11",
  "torch": "2.6.0+cu124",
  "cuda": "12.4",
  "transformers": "5.21.0",
  "sklearn": "1.9.1",
  "numpy": "2.5.4",
  "pandas": "3.1.0",
  "scipy": "1.18.1",
  "gpu": "NVIDIA GeForce RTX 4070 SUPER",
  "scibert_snapshot": "24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1"
}


## 1. Utilidades

In [3]:
def fijar_semillas(semilla: int) -> None:
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)
    torch.cuda.manual_seed_all(semilla)


def liberar_gpu() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def sha256_bytes(datos: bytes) -> str:
    return hashlib.sha256(datos).hexdigest()


def sha_ids(ids) -> str:
    return sha256_bytes(json.dumps(sorted(ids)).encode("utf-8"))


def escribir_json(ruta: Path, obj) -> None:
    ruta.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding="utf-8")


def leer_csv(ruta: Path, **kw) -> pd.DataFrame:
    return pd.read_csv(ruta, dtype=str, keep_default_na=False, encoding="utf-8-sig", **kw)


def leer_jsonl(ruta: Path) -> list[dict]:
    with open(ruta, encoding="utf-8") as f:
        return [json.loads(linea) for linea in f if linea.strip()]


def una_caliente(indice: int) -> list[float]:
    return [1.0 if i == indice else 0.0 for i in range(N_CLASES)]

## 2. Datos comunes a las dos pistas

In [4]:
CTX = json.loads((RAW / "contexts.json").read_text(encoding="utf-8-sig"))
PAPERS = json.loads((RAW / "papers.json").read_text(encoding="utf-8-sig"))

GOLD_SHA = sha256_bytes(GOLD_CSV.read_bytes())
assert GOLD_SHA.startswith(GOLD_SHA_PREFIJO), f"El gold cambio: sha256 {GOLD_SHA}"
GOLD_IDS = leer_csv(GOLD_CSV, usecols=["context_id"])["context_id"].tolist()   # solo ids: sin etiquetas
assert len(GOLD_IDS) == len(set(GOLD_IDS)) and all(c in CTX for c in GOLD_IDS)

KD = {}   # distribucion del QLoRA por contexto (solo la usa la pista M)
for ruta in PREDS_QLORA:
    for r in leer_jsonl(ruta):
        base = r["dist"][:N_CLASES]
        total = sum(base)
        KD[r["context_id"]] = [x / total for x in base] if total > 0 else [1 / N_CLASES] * N_CLASES
print("gold sha256:", GOLD_SHA[:16], "| gold:", len(GOLD_IDS), "| KD:", len(KD))

gold sha256: 80e4d9a6a72bd369 | gold: 1533 | KD: 61014


### 2.1 Fragmentación V2 y cobertura

In [5]:
_HEURISTICA = (
    ("introduction", ("introduc", "motivation")),
    ("related_work", ("related", "background", "prior work", "previous work")),
    ("conclusion", ("conclusion", "future work", "summary", "discussion")),
    ("results", ("result", "experiment", "evaluation", "analysis", "finding")),
    ("method", ("method", "approach", "model", "system", "algorithm", "framework", "data", "corpus", "task")),
)


def perfil_seccion(nombre: str | None) -> str:
    # Mapeo heuristico por palabras clave (no es una etiqueta retorica validada)
    if not nombre or not nombre.strip():
        return "vacia"
    n = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode().lower()
    for perfil, claves in _HEURISTICA:
        if any(k in n for k in claves):
            return perfil
    return "otra"


def leer_parrafos(cited: str) -> list[dict]:
    ruta = CACHE_OCL / f"{cited}.json"
    if not ruta.is_file():
        raise FileNotFoundError("ocl_404" if (CACHE_OCL / f"{cited}.404").exists() else "sin_descargar")
    doc = json.loads(ruta.read_text(encoding="utf-8"))
    if doc.get("paper_id") != cited:
        raise ValueError("id_ocl_distinto")
    parrafos = []
    for numero, p in enumerate(doc.get("pdf_parse", {}).get("body_text", [])):
        texto = str(p.get("text") or "").strip()
        if texto:
            parrafos.append({"numero": numero, "texto": texto, "seccion": str(p.get("section") or "").strip()})
    if not parrafos:
        raise ValueError("sin_parrafos")
    return parrafos


def integrar_contexto(context_id: str) -> dict:
    # Contexto ACL-200 + parrafos del articulo citado en ACL OCL
    ctx = CTX[context_id]
    if ctx["masked_text"].count("TARGETCIT") != 1:
        raise ValueError("targetcit_no_unico")
    if ctx["refid"] not in PAPERS:
        raise ValueError("citado_fuera_de_acl200")
    return {"context_id": context_id, "citing_id": ctx["citing_id"], "cited_id": ctx["refid"],
            "citation_context": ctx["masked_text"], "paragraphs": leer_parrafos(ctx["refid"])}


def fragmentar(parrafos: list[dict]) -> list[dict]:
    # V1: hasta 2 parrafos contiguos de la misma seccion, <= 300 palabras; un parrafo largo invalida el articulo
    chunks, pendientes, palabras, seccion_anterior = [], [], 0, None

    def cerrar() -> None:
        nonlocal pendientes, palabras
        if pendientes:
            chunks.append({"chunk_id": f"p{pendientes[0]['numero']:05d}-p{pendientes[-1]['numero']:05d}",
                           "texto": "\n".join(p["texto"] for p in pendientes), "seccion": seccion_anterior,
                           "paragraph_indices": [p["numero"] for p in pendientes], "palabras": palabras})
        pendientes, palabras = [], 0

    for p in parrafos:
        n = len(p["texto"].split())
        if n > MAX_PALABRAS or len(p["texto"]) > MAX_CARACTERES:
            raise ValueError("parrafo_largo")
        if pendientes and (len(pendientes) >= MAX_PARRAFOS_CHUNK or palabras + n > MAX_PALABRAS
                           or p["seccion"] != seccion_anterior or p["numero"] != pendientes[-1]["numero"] + 1):
            cerrar()
        pendientes.append(p)
        palabras += n
        seccion_anterior = p["seccion"]
    cerrar()
    if not chunks:
        raise ValueError("sin_fragmentos")
    return chunks


def _dividir_parrafo_largo_v2(texto: str) -> list[str]:
    # Divide solo por limites de tokens; prefiere cerrar en oracion si no desperdicia mas de media ventana
    tokens = list(re.finditer(r"\S+", texto))
    partes, inicio = [], 0
    while inicio < len(tokens):
        j, origen = inicio, tokens[inicio].start()
        while j < len(tokens) and j - inicio < MAX_PALABRAS and tokens[j].end() - origen <= MAX_CARACTERES:
            j += 1
        if j == inicio:
            raise ValueError("token_demasiado_largo")
        if j < len(tokens):
            cierres = [k for k in range(inicio + 1, j) if tokens[k - 1].group()[-1:] in (".", "!", "?")]
            if cierres and cierres[-1] - inicio >= (j - inicio) // 2:
                j = cierres[-1]
        partes.append(texto[origen:tokens[j - 1].end()].strip())
        inicio = j
    if " ".join(" ".join(p.split()) for p in partes) != " ".join(texto.split()):
        raise ValueError("v2_perdio_texto")
    return partes


def fragmentar_v2(parrafos: list[dict]) -> list[dict]:
    # V2: V1 para los tramos de parrafos cortos; segmentos pNNNNN-sMMM para los largos
    chunks, pendientes = [], []

    def cerrar_cortos() -> None:
        if pendientes:
            chunks.extend(fragmentar(pendientes))
            pendientes.clear()

    for p in parrafos:
        if len(p["texto"].split()) <= MAX_PALABRAS and len(p["texto"]) <= MAX_CARACTERES:
            pendientes.append(p)
            continue
        cerrar_cortos()
        partes = _dividir_parrafo_largo_v2(p["texto"])
        for i, parte in enumerate(partes):
            chunks.append({"chunk_id": f"p{p['numero']:05d}-s{i:03d}", "texto": parte, "seccion": p["seccion"],
                           "paragraph_indices": [p["numero"]], "palabras": len(parte.split())})
    cerrar_cortos()
    if not chunks:
        raise ValueError("sin_fragmentos")
    return chunks


def deduplicar_chunks(chunks: list[dict]) -> list[dict]:
    # Texto repetido (espacios colapsados): se conserva el de menor chunk_id, en el orden original
    mejor = {}
    for c in chunks:
        k = " ".join(c["texto"].split())
        if k not in mejor or c["chunk_id"] < mejor[k]:
            mejor[k] = c["chunk_id"]
    return [c for c in chunks if mejor[" ".join(c["texto"].split())] == c["chunk_id"]]

In [6]:
def procesar_citado(cited: str, context_id: str) -> dict:
    # Estado V1/V2 del articulo, chunks V2 deduplicados y parrafos largos (para la EDA)
    info = {"motivo_v1": None, "motivo_v2": None, "chunks": None, "v1_igual_v2": False, "largos": []}
    try:
        parrafos = integrar_contexto(context_id)["paragraphs"]
    except json.JSONDecodeError:
        info["motivo_v1"] = info["motivo_v2"] = "json_invalido"
        return info
    except (FileNotFoundError, ValueError) as exc:
        info["motivo_v1"] = info["motivo_v2"] = str(exc)
        return info
    info["largos"] = [{"cited_id": cited, "numero": p["numero"], "seccion": p["seccion"], "palabras": len(p["texto"].split()),
                       "caracteres": len(p["texto"]), "extracto": " ".join(p["texto"].split())[:300]}
                      for p in parrafos if len(p["texto"].split()) > MAX_PALABRAS or len(p["texto"]) > MAX_CARACTERES]
    v1 = None
    try:
        v1 = deduplicar_chunks(fragmentar(parrafos))
    except ValueError as exc:
        info["motivo_v1"] = str(exc)
    try:
        info["chunks"] = deduplicar_chunks(fragmentar_v2(parrafos))
    except ValueError as exc:
        info["motivo_v2"] = str(exc)
    info["v1_igual_v2"] = v1 is not None and v1 == info["chunks"]
    return info

In [7]:
def motivo_cobertura(fila: dict, info_citados: dict) -> str:
    # "ok" si la cita tiene entrada AB: articulo fragmentado, >= 3 chunks y titulo/abstract del citado
    if fila["texto"].count("TARGETCIT") != 1:
        return "targetcit_no_unico"
    info = info_citados[fila["cited_id"]]
    if info["motivo_v2"]:
        return info["motivo_v2"]
    if len(info["chunks"]) < TOP_K:
        return "menos_de_3_chunks"
    paper = PAPERS.get(fila["cited_id"]) or {}
    if not (paper.get("title") or "").strip() or not (paper.get("abstract") or "").strip():
        return "sin_titulo_abstract"
    return "ok"


def huella_chunks(chunks: list[dict]) -> str:
    return sha256_bytes(json.dumps([(c["chunk_id"], c["texto"]) for c in chunks], ensure_ascii=False).encode("utf-8"))

In [8]:
_MARCADORES = re.compile(r"\b(?:TARGETCIT|OTHERCIT)\b")
_PALABRAS = re.compile(r"[a-z0-9]+")


def shingles(texto: str, k: int = K_SHINGLE) -> set[int]:
    # Los marcadores cortan el texto: no se arman k-gramas que los crucen
    salida = set()
    for segmento in _MARCADORES.split(texto):
        palabras = _PALABRAS.findall(segmento.lower())
        for i in range(len(palabras) - k + 1):
            salida.add(hash(tuple(palabras[i:i + k])))
    return salida


def casi_duplicados(textos_a: dict[str, str], textos_b: dict[str, str], umbral: float = UMBRAL_SOLAPE) -> list[dict]:
    # Pares (a, b) con |comunes| / min(|a|, |b|) >= umbral; indice invertido sobre b
    huellas_b = {c: shingles(t) for c, t in textos_b.items()}
    indice = defaultdict(list)
    for c, s in huellas_b.items():
        for h in s:
            indice[h].append(c)
    pares = []
    for ca, ta in textos_a.items():
        s = shingles(ta)
        if not s:
            continue
        comunes = Counter()
        for h in s:
            comunes.update(indice.get(h, ()))
        for cb, n in comunes.items():
            if ca == cb:
                continue
            solape = n / min(len(s), len(huellas_b[cb]))
            if solape >= umbral:
                pares.append({"a": ca, "b": cb, "solape": round(solape, 4)})
    return pares


def textos(filas: list[dict]) -> dict[str, str]:
    return {x["context_id"]: x["texto"] for x in filas}

### 2.2 Entradas A / AB

In [9]:
TOKENIZADOR = AutoTokenizer.from_pretrained(SCIBERT)
TOKENIZADOR.model_max_length = int(1e9)   # el recorte lo hace `ensamblar`
CLS, SEP, PAD = TOKENIZADOR.cls_token_id, TOKENIZADOR.sep_token_id, TOKENIZADOR.pad_token_id
_CACHE_TOKENS: dict[str, list[int]] = {}


def pretokenizar(textos_: list[str]) -> None:
    nuevos = sorted({t for t in textos_ if t not in _CACHE_TOKENS})
    for ini in range(0, len(nuevos), 2048):
        lote = nuevos[ini:ini + 2048]
        for t, ids in zip(lote, TOKENIZADOR(lote, add_special_tokens=False)["input_ids"]):
            _CACHE_TOKENS[t] = ids


def ids_de(texto: str) -> list[int]:
    if texto not in _CACHE_TOKENS:
        pretokenizar([texto])
    return _CACHE_TOKENS[texto]


def limpio(texto: str) -> str:
    return " ".join(texto.split())


def segmentos(fila: dict, entrada: str, perfiles_top3: dict) -> list[dict]:
    paper = PAPERS[fila["cited_id"]]
    segs = [{"nombre": "contexto", "tipo": "contexto", "texto": limpio(fila["texto"])},
            {"nombre": "titulo", "tipo": "titulo", "texto": limpio(paper["title"])},
            {"nombre": "abstract", "tipo": "abstract", "texto": limpio(paper["abstract"])}]
    if entrada == "AB":
        segs += [{"nombre": f"chunk_{i}", "tipo": "chunk", "texto": f"{c['perfil']} : {limpio(c['texto'])}"}
                 for i, c in enumerate(perfiles_top3[fila["context_id"]], 1)]
    return segs


def tokenizar_segmentos(segs: list[dict]) -> list[dict]:
    salida = []
    for s in segs:
        if s["tipo"] == "contexto":
            antes, despues = s["texto"].split("TARGETCIT")
            a, m, z = ids_de(antes), ids_de("TARGETCIT"), ids_de(despues)
            salida.append({**s, "ids": a + m + z, "centro": (len(a), len(a) + len(m))})
        else:
            salida.append({**s, "ids": ids_de(s["texto"]), "centro": None})
    return salida


def reparto_water_filling(longitudes: list[int], presupuesto: int) -> list[int]:
    # Los segmentos cortos conservan todo; el sobrante se reparte entre los largos (resto a los primeros)
    asignado = [0] * len(longitudes)
    pendientes = sorted(range(len(longitudes)), key=lambda i: (longitudes[i], i))
    restante = presupuesto
    while pendientes:
        parte = restante // len(pendientes)
        i = pendientes[0]
        if longitudes[i] <= parte:
            asignado[i] = longitudes[i]
            restante -= longitudes[i]
            pendientes.pop(0)
            continue
        for j in pendientes:
            asignado[j] = parte
        restante -= parte * len(pendientes)
        for j in sorted(pendientes):
            if restante and asignado[j] < longitudes[j]:
                asignado[j] += 1
                restante -= 1
        break
    return asignado


def ventana_contexto(ids: list[int], centro: tuple[int, int], tope: int) -> list[int]:
    if len(ids) <= tope:
        return ids
    medio = (centro[0] + centro[1]) // 2
    inicio = min(max(medio - tope // 2, 0), len(ids) - tope)
    if centro[0] < inicio:
        inicio = centro[0]
    return ids[inicio:inicio + tope]


def ensamblar(segs: list[dict]) -> dict:
    # [CLS] ctx<=192 [SEP] titulo<=48 [SEP] abstract [SEP] (chunk [SEP]) x 3 con water-filling en el sobrante
    ctx, resto = segs[0], segs[1:]
    presupuesto = MAX_LEN - 2 - len(resto)
    ventana = ventana_contexto(ctx["ids"], ctx["centro"], min(TOPE_CONTEXTO, presupuesto))
    libre = presupuesto - len(ventana)
    titulo, abstract = resto[:2]
    t = min(len(titulo["ids"]), TOPE_TITULO, libre)
    a = min(len(abstract["ids"]), libre - t)
    asignado = {"titulo": t, "abstract": a}
    libre -= t + a
    chunks = resto[2:]
    if chunks:
        asignado.update({s["nombre"]: n for s, n in zip(chunks, reparto_water_filling([len(s["ids"]) for s in chunks], libre))})
    ids = [CLS, *ventana, SEP]
    truncados = {"contexto": len(ctx["ids"]) - len(ventana)}
    for s in resto:
        n = asignado[s["nombre"]]
        ids += [*s["ids"][:n], SEP]
        truncados[s["nombre"]] = len(s["ids"]) - n
    assert len(ids) <= MAX_LEN
    return {"input_ids": ids, "truncados": truncados, "longitudes": {s["nombre"]: len(s["ids"]) for s in segs}}


def construir_ejemplos(filas: list[dict], entrada: str, perfiles_top3: dict) -> list[dict]:
    textos_ = ["TARGETCIT"]
    for x in filas:
        for s in segmentos(x, entrada, perfiles_top3):
            textos_ += s["texto"].split("TARGETCIT") if s["tipo"] == "contexto" else [s["texto"]]
    pretokenizar(textos_)
    ejemplos = []
    for x in filas:
        r = ensamblar(tokenizar_segmentos(segmentos(x, entrada, perfiles_top3)))
        ejemplos.append({"context_id": x["context_id"], "citing_id": x["citing_id"], "input_ids": r["input_ids"],
                         "truncados": r["truncados"], "longitudes": r["longitudes"],
                         "indice": x.get("indice"), "objetivo": x.get("objetivo")})
    return ejemplos

## 3. Preparación de los datos por pista

Misma secuencia que 08/09 (split dev, L1, L2, L10, L4, cobertura AB, Top-3, entradas) y el mismo orden. El Top-3 se lee de
`experimentos_08_09` y se valida con la huella de los chunks. Los `sha_ids` de train, dev y V deben coincidir con los de la corrida elegida.

In [10]:
def filas_de(df: pd.DataFrame, con_kd: bool) -> list[dict]:
    # Conserva el orden de las filas del CSV
    filas = []
    for r in df.itertuples(index=False):
        ctx = CTX[r.context_id]
        assert ctx["citing_id"] == r.citing_id, r.context_id
        indice = INDICE[r.etiqueta_entrenamiento]
        objetivo = una_caliente(indice)
        if con_kd and r.fuente == "qlora":
            objetivo = KD[r.context_id]
        filas.append({"context_id": r.context_id, "citing_id": r.citing_id, "cited_id": ctx["refid"],
                      "texto": ctx["masked_text"], "indice": indice, "objetivo": objetivo, "fuente": r.fuente,
                      "p": float(r.p) if r.p else None, "nivel": r.nivel})
    return filas


def filas_sin_etiqueta(ids: list[str]) -> list[dict]:
    return [{"context_id": c, "citing_id": CTX[c]["citing_id"], "cited_id": CTX[c]["refid"], "texto": CTX[c]["masked_text"]}
            for c in ids]


def en_dev(citing_id: str) -> bool:
    h = int(hashlib.sha256(f"{SAL_DEV}:{citing_id}".encode("utf-8")).hexdigest(), 16)
    return h % 100 < round(100 * FRACCION_DEV)


ORDEN = ("train", "dev", "V", "gold")

# ---------------------------------------------------------------- L1
CLAVES_L1 = ("context_id", "citing_id", "par", "sha_texto")


def valor_l1(fila: dict, clave: str):
    if clave == "par":
        return (fila["citing_id"], fila["cited_id"])
    if clave == "sha_texto":
        return sha256_bytes(" ".join(fila["texto"].split()).encode("utf-8"))
    return fila[clave]


def claves_l1(filas: list[dict]) -> dict[str, set]:
    return {k: {valor_l1(x, k) for x in filas} for k in CLAVES_L1}


def fugas_l1(conj: dict[str, list[dict]]) -> dict[str, int]:
    indices = {n: claves_l1(conj[n]) for n in ORDEN}
    return {f"{a}~{b}:{k}": len(indices[a][k] & indices[b][k])
            for i, a in enumerate(ORDEN) for b in ORDEN[i + 1:] for k in CLAVES_L1}


def quitar_l1(filas: list[dict], otras: list[dict]) -> list[dict]:
    prohibidas = claves_l1(otras)
    return [x for x in filas if not any(valor_l1(x, k) in prohibidas[k] for k in CLAVES_L1)]


# ---------------------------------------------------------------- L10
def textos(filas: list[dict]) -> dict[str, str]:
    return {x["context_id"]: x["texto"] for x in filas}


def correr_l10(conj: dict[str, list[dict]]) -> dict[str, list[dict]]:
    return {f"{a}~{b}": casi_duplicados(textos(conj[a]), textos(conj[b]))
            for i, a in enumerate(ORDEN) for b in ORDEN[i + 1:]}


# ---------------------------------------------------------------- L4
def indice_contextos(filas: list[dict]) -> tuple[dict[int, list[str]], dict[str, int]]:
    indice, tamanos = defaultdict(list), {}
    for x in filas:
        s = shingles(x["texto"])
        tamanos[x["context_id"]] = len(s)
        for h in s:
            indice[h].append(x["context_id"])
    return indice, tamanos


def solapes_l4(filas_chunks: list[dict], filas_eval: list[dict], info_citados: dict) -> list[dict]:
    # Todos los chunks (V2) de los citados de `filas_chunks` contra los contextos de `filas_eval`
    indice, tamanos = indice_contextos(filas_eval)
    universo = set(indice)
    violaciones = []
    for cited in sorted({x["cited_id"] for x in filas_chunks}):
        for c in info_citados[cited]["chunks"] or []:
            s = shingles(c["texto"])
            comunes = s & universo
            if not comunes:
                continue
            cuenta = Counter()
            for h in comunes:
                cuenta.update(indice[h])
            for cid, n in cuenta.items():
                solape = n / min(len(s), tamanos[cid])
                if solape >= UMBRAL_SOLAPE:
                    violaciones.append({"cited_id": cited, "chunk_id": c["chunk_id"], "contexto_eval": cid, "solape": round(solape, 4)})
    return violaciones


def correr_l4(conj: dict[str, list[dict]], info_citados: dict) -> dict[str, list[dict]]:
    return {"train~V+gold+dev": solapes_l4(conj["train"], conj["V"] + conj["gold"] + conj["dev"], info_citados),
            "dev~V+gold": solapes_l4(conj["dev"], conj["V"] + conj["gold"], info_citados),
            "V~gold": solapes_l4(conj["V"], conj["gold"], info_citados)}


def preparar(pista: str) -> dict:
    cfg, modelo = PISTAS[pista], MODELOS[pista]
    assert modelo["entrada"] == "AB"
    ref = json.loads((ORIGEN_08_09 / pista / "corridas" / f"{nombre_corrida(pista, modelo)}.json").read_text(encoding="utf-8"))
    t = time.time()

    # ---- split train / dev / V / gold
    train_df, v_df = leer_csv(DATOS / cfg["train"]), leer_csv(DATOS / cfg["v"])
    assert train_df["context_id"].is_unique and v_df["context_id"].is_unique
    if cfg["kd"]:
        faltan = [c for c, f in zip(train_df["context_id"], train_df["fuente"]) if f == "qlora" and c not in KD]
        assert not faltan, f"{len(faltan)} filas qlora sin distribucion KD"
    train_todo, gold_filas = filas_de(train_df, cfg["kd"]), filas_sin_etiqueta(GOLD_IDS)
    conj = {"train": [x for x in train_todo if not en_dev(x["citing_id"])],
            "dev": [x for x in train_todo if en_dev(x["citing_id"])],
            "V": filas_de(v_df, False), "gold": gold_filas}
    controles = []

    def registrar(control: str, nombre: str, antes: int) -> None:
        controles.append({"control": control, "conjunto": nombre, "antes": antes, "despues": len(conj[nombre])})

    # ---- L1 y L2
    inicial = fugas_l1(conj)
    assert all(v == 0 for k, v in inicial.items() if k.startswith("V~gold")), "V y gold comparten casos"
    for nombre, posteriores in (("train", ("dev", "V", "gold")), ("dev", ("V", "gold"))):
        antes = len(conj[nombre])
        conj[nombre] = quitar_l1(conj[nombre], [x for p in posteriores for x in conj[p]])
        registrar("L1", nombre, antes)
    assert sum(fugas_l1(conj).values()) == 0
    citantes_eval = {x["citing_id"] for n in ("dev", "V", "gold") for x in conj[n]}
    for nombre in ("train", "dev"):
        antes = len(conj[nombre])
        conj[nombre] = [x for x in conj[nombre] if x["cited_id"] not in citantes_eval]
        registrar("L2", nombre, antes)

    # ---- L10
    l10 = correr_l10(conj)
    quitar = defaultdict(set)
    for par, detalle in l10.items():
        quitar[par.split("~")[0]] |= {d["a"] for d in detalle}   # nunca se quita del gold
    for nombre in ("train", "dev", "V"):
        antes = len(conj[nombre])
        conj[nombre] = [x for x in conj[nombre] if x["context_id"] not in quitar[nombre]]
        registrar("L10", nombre, antes)
    assert sum(len(d) for d in correr_l10(conj).values()) == 0
    assert sum(fugas_l1(conj).values()) == 0

    # ---- fragmentos de los citados (V2)
    primer_contexto = {}
    for x in sorted([x for n in ("train", "dev", "V") for x in conj[n]] + gold_filas, key=lambda x: x["context_id"]):
        primer_contexto.setdefault(x["cited_id"], x["context_id"])
    info_citados = {c: procesar_citado(c, primer_contexto[c]) for c in sorted(primer_contexto)}
    assert all(i["v1_igual_v2"] for i in info_citados.values() if i["motivo_v1"] is None)

    # ---- L4
    l4 = correr_l4(conj, info_citados)
    citados_l4 = {v["cited_id"] for vs in l4.values() for v in vs}
    for nombre in ("train", "dev", "V"):
        antes = len(conj[nombre])
        conj[nombre] = [x for x in conj[nombre] if x["cited_id"] not in citados_l4]
        registrar("L4", nombre, antes)
    assert sum(len(v) for v in correr_l4(conj, info_citados).values()) == 0

    # ---- cobertura AB
    for nombre in ("train", "dev", "V"):
        antes = len(conj[nombre])
        conj[nombre] = [x for x in conj[nombre] if motivo_cobertura(x, info_citados) == "ok"]
        registrar("cobertura_AB", nombre, antes)
    cob_gold = {x["context_id"] for x in gold_filas if motivo_cobertura(x, info_citados) == "ok"}
    assert len(cob_gold) / len(gold_filas) >= 0.99

    # ---- Top-3 de 08/09 (no se carga el recuperador)
    top3 = json.loads((ORIGEN_08_09 / pista / "top3_v2.json").read_text(encoding="utf-8"))
    contextos = {x["context_id"]: x for n in ("train", "dev", "V") for x in conj[n]}
    contextos.update({x["context_id"]: x for x in gold_filas if x["context_id"] in cob_gold})
    perfiles_top3 = {}
    for c, x in contextos.items():
        assert c in top3, f"{c}: sin Top-3 en 08/09"
        chunks = info_citados[x["cited_id"]]["chunks"]
        assert top3[c]["huella"] == huella_chunks(chunks), f"{c}: la huella de los chunks no coincide"
        por_id = {ch["chunk_id"]: ch for ch in chunks}
        perfiles_top3[c] = [{"chunk_id": t3["chunk_id"], "texto": por_id[t3["chunk_id"]]["texto"],
                             "seccion": por_id[t3["chunk_id"]]["seccion"], "perfil": perfil_seccion(por_id[t3["chunk_id"]]["seccion"]),
                             "similitud": t3["similitud"]} for t3 in top3[c]["top3"]]

    # ---- entradas AB
    gold_ab = [x for x in gold_filas if x["context_id"] in cob_gold]
    ejemplos = {n: construir_ejemplos(conj[n], "AB", perfiles_top3) for n in ("train", "dev", "V")}
    ejemplos["gold"] = construir_ejemplos(gold_ab, "AB", perfiles_top3)
    sha = {n: sha_ids(e["context_id"] for e in ejemplos[n]) for n in ("train", "dev", "V")}
    assert sha == ref["sha_ids"], {n: (sha[n][:12], ref["sha_ids"][n][:12]) for n in sha}
    assert {n: len(ejemplos[n]) for n in ORDEN} == ref["n"], ref["n"]
    conteo_train = np.bincount([x["indice"] for x in conj["train"]], minlength=N_CLASES)
    print(f"[{pista}] datos listos en {time.time() - t:.0f}s | n:", {n: len(e) for n, e in ejemplos.items()},
          "| sha_ids = 08/09 | huellas Top-3 ok:", len(perfiles_top3))
    return {"ejemplos": ejemplos, "conteo_train": conteo_train, "cob_gold": cob_gold, "sha_ids": sha, "controles": controles}

## 4. Modelo y entrenamiento

In [11]:
def nombre_corrida(pista: str, m: dict) -> str:
    return f"{pista}-{m['entrada']}-{m['estrategia']}-lr{m['lr']:g}-s{m['semilla']}"


def hacer_perdida(estrategia: str, conteo_train: np.ndarray):
    # E0: CE sin pesos | E3: focal (gamma=2) con pesos class-balanced (beta=0,999) calculados sobre `conteo_train`
    assert estrategia in ("E0", "E3"), estrategia
    conteo = conteo_train.astype(float)
    cb = (1 - BETA_CB) / (1 - BETA_CB ** np.maximum(conteo, 1))
    alfa = torch.tensor(cb / cb.sum() * N_CLASES, dtype=torch.float32, device=DISPOSITIVO)

    def perdida(logits: torch.Tensor, objetivo: torch.Tensor) -> torch.Tensor:
        logp = torch.log_softmax(logits.float(), dim=-1)
        if estrategia == "E3":
            por_muestra = -(objetivo * alfa * (1 - logp.exp()) ** GAMMA_FOCAL * logp).sum(-1)
            w = (objetivo * alfa).sum(-1)
            return por_muestra.sum() / w.sum()
        return (-(objetivo * logp).sum(-1)).mean()

    return perdida

In [12]:
class ClasificadorCLS(nn.Module):
    def __init__(self, semilla: int):
        super().__init__()
        # atencion "eager": SDPA tiene retroceso no determinista en CUDA
        self.encoder = AutoModel.from_pretrained(SCIBERT, attn_implementation="eager")
        torch.manual_seed(semilla)   # la cabeza se inicializa con la semilla de la corrida
        self.dropout = nn.Dropout(DROPOUT)
        self.cabeza = nn.Linear(self.encoder.config.hidden_size, N_CLASES)

    def forward(self, input_ids, attention_mask):
        cls = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0]
        return self.cabeza(self.dropout(cls))


def lote_tensores(ejemplos: list[dict]) -> tuple[torch.Tensor, torch.Tensor]:
    largo = max(len(e["input_ids"]) for e in ejemplos)
    ids = torch.full((len(ejemplos), largo), PAD, dtype=torch.long)
    mask = torch.zeros((len(ejemplos), largo), dtype=torch.long)
    for i, e in enumerate(ejemplos):
        ids[i, :len(e["input_ids"])] = torch.tensor(e["input_ids"])
        mask[i, :len(e["input_ids"])] = 1
    return ids.to(DISPOSITIVO), mask.to(DISPOSITIVO)


def autocast():
    return torch.autocast(device_type=DISPOSITIVO.type, dtype=torch.bfloat16, enabled=DISPOSITIVO.type == "cuda")


@torch.inference_mode()
def logits_de(modelo: nn.Module, ejemplos: list[dict]) -> np.ndarray:
    # Inferencia por lotes ordenados por longitud; devuelve los logits en el orden de `ejemplos`
    modelo.eval()
    orden = sorted(range(len(ejemplos)), key=lambda i: len(ejemplos[i]["input_ids"]))
    salida = np.zeros((len(ejemplos), N_CLASES), dtype=np.float32)
    for ini in range(0, len(orden), BATCH_INFERENCIA):
        idx = orden[ini:ini + BATCH_INFERENCIA]
        ids, mask = lote_tensores([ejemplos[i] for i in idx])
        with autocast():
            salida[idx] = modelo(ids, mask).float().cpu().numpy()
    return salida


def matriz_confusion(y: np.ndarray, pred: np.ndarray) -> np.ndarray:
    return np.bincount(np.asarray(y) * N_CLASES + np.asarray(pred), minlength=N_CLASES ** 2).reshape(N_CLASES, N_CLASES)


def f1_por_clase(mat: np.ndarray) -> np.ndarray:
    # zero_division = 0; acepta matrices apiladas (..., 9, 9)
    tp = np.diagonal(mat, axis1=-2, axis2=-1).astype(float)
    den = mat.sum(-1) + mat.sum(-2)
    return np.divide(2 * tp, den, out=np.zeros_like(tp), where=den > 0)


def metricas(y, pred) -> dict:
    # macro sobre las 9 clases (tambien las sin soporte), micro = accuracy, ponderado por soporte
    mat = matriz_confusion(y, pred)
    f1 = f1_por_clase(mat)
    soporte = mat.sum(1)
    n = mat.sum()
    return {"n": int(n), "accuracy": float(np.trace(mat) / n), "f1_macro": float(f1.mean()), "f1_micro": float(np.trace(mat) / n),
            "f1_weighted": float((f1 * soporte).sum() / n), "f1_clase": f1.tolist(), "soporte": soporte.tolist(),
            "confusion": mat.tolist()}


def ajustar_sesgos(logits: np.ndarray, y: np.ndarray) -> np.ndarray:
    # Ascenso coordenado sobre un sesgo por clase (sumado a log-softmax) que maximiza el F1-macro de dev
    logp = logits - np.logaddexp.reduce(logits, axis=1, keepdims=True)
    sesgos = np.zeros(N_CLASES)
    mejor = metricas(y, (logp + sesgos).argmax(1))["f1_macro"]
    for _ in range(RONDAS_SESGOS):
        cambio = False
        for c in range(N_CLASES):
            for v in GRILLA_SESGOS:
                prueba = sesgos.copy()
                prueba[c] = v
                f1 = metricas(y, (logp + prueba).argmax(1))["f1_macro"]
                if f1 > mejor + 1e-12:
                    mejor, sesgos, cambio = f1, prueba, True
        if not cambio:
            break
    return sesgos


def predecir(logits: np.ndarray, sesgos: np.ndarray | None = None) -> np.ndarray:
    if sesgos is None:
        return logits.argmax(1)
    return (logits - np.logaddexp.reduce(logits, axis=1, keepdims=True) + sesgos).argmax(1)

In [13]:
def entrenar(m: dict, datos: dict) -> tuple[nn.Module, dict]:
    lr, semilla = m["lr"], m["semilla"]
    perdida = hacer_perdida(m["estrategia"], datos["conteo_train"])
    train, dev = datos["ejemplos"]["train"], datos["ejemplos"]["dev"]
    y_dev = np.array([e["indice"] for e in dev])
    fijar_semillas(semilla)
    modelo = ClasificadorCLS(semilla).to(DISPOSITIVO)
    fijar_semillas(semilla)   # como #104: la semilla se vuelve a fijar despues de construir el modelo
    sin_decay = ("bias", "LayerNorm.weight")
    grupos = [{"params": [p for n, p in modelo.named_parameters() if not any(k in n for k in sin_decay)], "weight_decay": WEIGHT_DECAY},
              {"params": [p for n, p in modelo.named_parameters() if any(k in n for k in sin_decay)], "weight_decay": 0.0}]
    optimizador = torch.optim.AdamW(grupos, lr=lr)
    lotes_por_epoca = math.ceil(len(train) / BATCH)
    total = math.ceil(lotes_por_epoca / ACUMULACION) * EPOCAS
    plan = get_linear_schedule_with_warmup(optimizador, round(WARMUP * total), total)
    generador = torch.Generator().manual_seed(semilla)
    cargador = DataLoader(train, batch_size=BATCH, shuffle=True, generator=generador, collate_fn=lambda b: b)
    historial, mejor_f1, mejor_estado, mejor_epoca, sin_mejora = [], -1.0, None, 0, 0
    for epoca in range(1, EPOCAS + 1):
        modelo.train()
        t0, acumulado = time.time(), 0.0
        optimizador.zero_grad(set_to_none=True)
        for i, lote in enumerate(cargador, 1):
            ids, mask = lote_tensores(lote)
            objetivo = torch.tensor([e["objetivo"] for e in lote], dtype=torch.float32, device=DISPOSITIVO)
            with autocast():
                logits = modelo(ids, mask)
            valor = perdida(logits, objetivo)
            (valor / ACUMULACION).backward()
            acumulado += float(valor.detach())
            if i % ACUMULACION == 0 or i == lotes_por_epoca:
                torch.nn.utils.clip_grad_norm_(modelo.parameters(), CLIP)
                optimizador.step()
                plan.step()
                optimizador.zero_grad(set_to_none=True)
        f1_dev = metricas(y_dev, logits_de(modelo, dev).argmax(1))["f1_macro"]
        historial.append({"epoca": epoca, "loss_train": acumulado / lotes_por_epoca, "f1_macro_dev": f1_dev,
                          "segundos": round(time.time() - t0, 1)})
        print(f"    epoca {epoca}: loss {historial[-1]['loss_train']:.4f} | F1-macro dev {f1_dev:.4f} | {historial[-1]['segundos']:.0f}s")
        if f1_dev > mejor_f1:
            mejor_f1, mejor_epoca, sin_mejora = f1_dev, epoca, 0
            mejor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
        else:
            sin_mejora += 1
            if sin_mejora >= PACIENCIA:
                break
    modelo.load_state_dict(mejor_estado)
    return modelo, {"historial": historial, "mejor_epoca": mejor_epoca, "epocas_ejecutadas": len(historial), "f1_macro_dev": mejor_f1}

## 5. Entrenar y guardar

Para cada pista: entrena, calcula logits de dev, V y gold, ajusta los sesgos PH sobre dev (si corresponde), guarda `model.safetensors`,
el manifiesto y las predicciones del gold, y recarga el archivo en un modelo nuevo para comprobar que reproduce los logits.

In [14]:
def sha256_archivo(ruta: Path) -> str:
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()


def entrenar_y_guardar(pista: str, datos: dict) -> dict:
    m = MODELOS[pista]
    nombre = nombre_corrida(pista, m)
    carpeta = SALIDA / pista
    carpeta.mkdir(parents=True, exist_ok=True)
    ej = datos["ejemplos"]
    print(f"[entrenando] {nombre}")
    t0 = time.time()
    modelo, info = entrenar(m, datos)
    logits = {n: logits_de(modelo, ej[n]) for n in ("dev", "V", "gold")}
    y = {n: np.array([e["indice"] for e in ej[n]]) for n in ("dev", "V")}
    sesgos = ajustar_sesgos(logits["dev"], y["dev"]) if m["ph"] else None
    m_v = metricas(y["V"], predecir(logits["V"], sesgos))
    print(f"    F1-macro V ({'con' if m['ph'] else 'sin'} PH): {m_v['f1_macro']:.4f}")

    # ---- pesos
    ruta_pesos = carpeta / "model.safetensors"
    save_file({k: v.detach().cpu().contiguous() for k, v in modelo.state_dict().items()}, str(ruta_pesos))
    del modelo
    liberar_gpu()

    # ---- predicciones ciegas del gold
    ids_gold = [e["context_id"] for e in ej["gold"]]
    pred, pred_ph = predecir(logits["gold"]), predecir(logits["gold"], sesgos)
    probs = torch.softmax(torch.tensor(logits["gold"]), dim=-1).numpy()
    with open(carpeta / "pred_gold.jsonl", "w", encoding="utf-8") as f:
        for c, p, pp, pr in zip(ids_gold, pred, pred_ph, probs):
            f.write(json.dumps({"context_id": c, "pred": int(p), "pred_ph": int(pp), "probs": [round(float(x), 6) for x in pr]}) + "\n")

    # ---- el archivo guardado debe reproducir los logits del modelo en memoria
    recargado = ClasificadorCLS(m["semilla"]).to(DISPOSITIVO)
    recargado.load_state_dict(load_file(str(ruta_pesos)))
    logits_recarga = logits_de(recargado, ej["gold"])
    dif_recarga = float(np.abs(logits_recarga - logits["gold"]).max())
    assert dif_recarga < 1e-3, f"{nombre}: el modelo recargado difiere de memoria (max |dif| = {dif_recarga})"
    del recargado
    liberar_gpu()
    print(f"    recarga desde model.safetensors: max |dif| de logits gold = {dif_recarga:.2e}")

    escribir_json(carpeta / "manifiesto_entrenamiento.json", {
        "nombre": nombre, "pista": pista, "config": m, "sha_ids": datos["sha_ids"],
        "n": {n: len(e) for n, e in ej.items()}, "conteo_train": datos["conteo_train"].tolist(),
        "sesgos_ph": None if sesgos is None else sesgos.tolist(), "historial": info["historial"],
        "mejor_epoca": info["mejor_epoca"], "epocas_ejecutadas": info["epocas_ejecutadas"], "f1_macro_dev": info["f1_macro_dev"],
        "f1_macro_v": m_v["f1_macro"], "dif_max_logits_recarga": dif_recarga, "segundos": round(time.time() - t0, 1),
        "entorno": ENTORNO, "scibert_snapshot": SCIBERT.name, "sha256_model_safetensors": sha256_archivo(ruta_pesos)})
    return {"ids_gold": ids_gold, "logits_gold": logits_recarga, "sesgos": sesgos, "cob_gold": datos["cob_gold"]}

## 6. Ejecución

In [15]:
RESULTADOS = {}
for pista in ("M", "S"):
    datos = preparar(pista)
    RESULTADOS[pista] = entrenar_y_guardar(pista, datos)
    del datos
    liberar_gpu()

[M] datos listos en 34s | n: {'train': 10206, 'dev': 1813, 'V': 3315, 'gold': 1532} | sha_ids = 08/09 | huellas Top-3 ok: 16866
[entrenando] M-AB-E0-lr3e-05-s2026


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 44401.88it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

    epoca 1: loss 0.9707 | F1-macro dev 0.6138 | 226s
    epoca 2: loss 0.5874 | F1-macro dev 0.7216 | 223s
    epoca 3: loss 0.4657 | F1-macro dev 0.7156 | 217s
    epoca 4: loss 0.4103 | F1-macro dev 0.7241 | 228s
    F1-macro V (con PH): 0.6838


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 34859.11it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

    recarga desde model.safetensors: max |dif| de logits gold = 0.00e+00
[S] datos listos en 27s | n: {'train': 10038, 'dev': 1814, 'V': 3318, 'gold': 1532} | sha_ids = 08/09 | huellas Top-3 ok: 16702
[entrenando] S-AB-E3-lr2e-05-s42


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 48788.08it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

    epoca 1: loss 0.8145 | F1-macro dev 0.6783 | 230s
    epoca 2: loss 0.2826 | F1-macro dev 0.7144 | 227s
    epoca 3: loss 0.1416 | F1-macro dev 0.7213 | 224s
    epoca 4: loss 0.0662 | F1-macro dev 0.7265 | 229s
    F1-macro V (sin PH): 0.6634


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21383.06it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

    recarga desde model.safetensors: max |dif| de logits gold = 0.00e+00


## 7. Evaluación en el test gold

Mismos subconjuntos que 08/09 (`todo_cob`, `humano_cob`, `concordancia_cob`), con PH si la pista lo usa. Se compara con el valor de la
corrida original (CSV por semilla) y con los logits de gold guardados en el `.npz`.

In [16]:
GOLD_DF = pd.read_csv(GOLD_CSV, encoding="utf-8-sig")
assert sha256_bytes(GOLD_CSV.read_bytes()) == GOLD_SHA
ETIQ = {r.context_id: int(r.indice) for r in GOLD_DF.itertuples() if r.evaluable}
FUENTE = dict(zip(GOLD_DF["context_id"], GOLD_DF["fuente"]))
SUBS_BASE = {"todo": set(ETIQ), "humano": {c for c in ETIQ if FUENTE[c] != "solo_opus"},
             "concordancia": {c for c in ETIQ if FUENTE[c] == "concordancia"}}
assert (len(SUBS_BASE["todo"]), len(SUBS_BASE["humano"]), len(SUBS_BASE["concordancia"])) == (1531, 1061, 186)

filas = []
for pista, r in RESULTADOS.items():
    m = MODELOS[pista]
    nombre = nombre_corrida(pista, m)
    pred = dict(zip(r["ids_gold"], predecir(r["logits_gold"], r["sesgos"])))
    ref = pd.read_csv(EVIDENCIAS_08_09 / f"{pista}_gold_metricas_por_semilla.csv")
    ref = ref[(ref["modelo"] == "mejor-AB") & (ref["semilla"] == m["semilla"])].set_index("subconjunto")
    npz = np.load(ORIGEN_08_09 / pista / "corridas" / f"{nombre}.npz")
    assert list(npz["ids_gold"]) == r["ids_gold"], f"{nombre}: el orden de ids del gold cambio"
    dif_logits = float(np.abs(r["logits_gold"] - npz["logits_gold"]).max())
    for sub, ids_base in SUBS_BASE.items():
        ids = sorted(ids_base & r["cob_gold"])
        met = metricas([ETIQ[c] for c in ids], [pred[c] for c in ids])
        viejo = ref.loc[f"{sub}_cob"]
        filas.append({"pista": pista, "corrida": nombre, "ph": m["ph"], "subconjunto": f"{sub}_cob", "n": met["n"],
                      **{k: met[k] for k in ("f1_macro", "f1_micro", "f1_weighted", "accuracy")},
                      **{f"{k}_08_09": viejo[k] for k in ("f1_macro", "f1_micro", "f1_weighted", "accuracy")},
                      "dif_f1_macro": met["f1_macro"] - viejo["f1_macro"], "max_dif_logits_gold": dif_logits})
GOLD_METRICAS = pd.DataFrame(filas)
GOLD_METRICAS.to_csv(EVIDENCIAS / "gold_metricas.csv", index=False)
print(GOLD_METRICAS.round(4).to_string(index=False))

pista               corrida    ph      subconjunto    n  f1_macro  f1_micro  f1_weighted  accuracy  f1_macro_08_09  f1_micro_08_09  f1_weighted_08_09  accuracy_08_09  dif_f1_macro  max_dif_logits_gold
    M M-AB-E0-lr3e-05-s2026  True         todo_cob 1530    0.5512    0.6856       0.6723    0.6856          0.5512          0.6856             0.6723          0.6856           0.0                  0.0
    M M-AB-E0-lr3e-05-s2026  True       humano_cob 1060    0.5577    0.6887       0.6722    0.6887          0.5577          0.6887             0.6722          0.6887           0.0                  0.0
    M M-AB-E0-lr3e-05-s2026  True concordancia_cob  186    0.7335    0.8011       0.7906    0.8011          0.7335          0.8011             0.7906          0.8011           0.0                  0.0
    S   S-AB-E3-lr2e-05-s42 False         todo_cob 1530    0.6046    0.7255       0.7150    0.7255          0.6046          0.7255             0.7150          0.7255           0.0                 